
## MySQL Database Design & Ingestion

This notebook creates the local MySQL analytical database and loads the validated datasets produced in Stage 2.

### Workflow

**Processed CSVs → Campaign Dimension → MySQL Schema → Data Load → Verification**

### Tables

- `dim_campaign`
- `fact_lead_acquisition`
- `fact_funnel_event`

The notebook intentionally keeps database creation and ingestion separate from business analysis.  
Stage 4 will contain the analytical SQL queries and views.


## 1. Imports and Project Paths

In [3]:
from pathlib import Path
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL


ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROCESSED_DATA_DIR = ROOT_DIR / "data" / "processed"
SQL_DIR = ROOT_DIR / "sql"

LEADS_FILE = PROCESSED_DATA_DIR / "clean_leads.csv"
EVENTS_FILE = PROCESSED_DATA_DIR / "clean_funnel_events.csv"
SCHEMA_FILE = SQL_DIR / "01_schema.sql"

MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_DATABASE = "growth_analytics"

load_dotenv(ROOT_DIR / ".env")

MYSQL_USER = os.getenv("MYSQL_USER")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD")

print(f"Project root: {ROOT_DIR}")
print(f"Database: {MYSQL_DATABASE}")


Project root: c:\Projects\b2b-growth-funnel-analytics
Database: growth_analytics


## 2. Validate Required Files and Credentials

The notebook should stop immediately if Stage 2 outputs or MySQL credentials are missing.


In [4]:
required_files = [
    LEADS_FILE,
    EVENTS_FILE,
    SCHEMA_FILE,
]

missing_files = [
    path for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Missing required files:\n"
        + "\n".join(str(path) for path in missing_files)
    )

if not MYSQL_USER or not MYSQL_PASSWORD:
    raise ValueError(
        "MYSQL_USER and MYSQL_PASSWORD must be defined "
        "in the project .env file."
    )

print("Required files and credentials are available.")


Required files and credentials are available.


## 3. Load Processed Data

In [5]:
leads = pd.read_csv(
    LEADS_FILE,
    parse_dates=["created_date"],
)

events = pd.read_csv(
    EVENTS_FILE,
    parse_dates=["stage_date"],
)

print(f"Processed leads:  {leads.shape}")
print(f"Processed events: {events.shape}")


Processed leads:  (500, 8)
Processed events: (1445, 5)


In [6]:
leads.head()

,lead_id,created_date,channel,campaign,region,company_size,industry,acquisition_cost
0,L00052,2025-01-01,Referral,Partner Referral,Canada,Mid-Market,SaaS,10.89
1,L00109,2025-01-01,Google Ads,Search Campaign,US,SMB,Logistics,66.52
2,L00197,2025-01-01,Webinar,Finance Automation Webinar,Canada,Mid-Market,Finance,65.79
3,L00327,2025-01-01,Google Ads,Search Campaign,Canada,SMB,SaaS,96.07
4,L00413,2025-01-02,Email,Product Update,US,Mid-Market,SaaS,13.52


In [7]:
events.head()

,event_id,lead_id,stage,stage_date,revenue
0,EVT000001,L00001,Lead,2025-03-25,0.0
1,EVT000002,L00001,Lost,2025-04-04,0.0
2,EVT000003,L00002,Lead,2025-06-11,0.0
3,EVT000004,L00002,MQL,2025-06-24,0.0
4,EVT000005,L00002,Lost,2025-07-13,0.0


## 4. Build Campaign Dimension

`campaign` and `channel` are repeated across lead records.  
We normalize them into a small campaign dimension and replace the repeated text in the lead fact table with `campaign_id`.


In [8]:
campaign_dim = (
    leads[["campaign", "channel"]]
    .drop_duplicates()
    .sort_values(["channel", "campaign"])
    .reset_index(drop=True)
)

campaign_dim.insert(
    0,
    "campaign_id",
    range(1, len(campaign_dim) + 1),
)

campaign_dim = campaign_dim.rename(
    columns={"campaign": "campaign_name"}
)

campaign_dim


,campaign_id,campaign_name,channel
0,1,Nurture Flow,Email
1,2,Product Update,Email
2,3,Reactivation,Email
3,4,Brand Campaign,Google Ads
4,5,Competitor Terms,Google Ads
5,6,Search Campaign,Google Ads
6,7,ABM Decision Makers,LinkedIn
7,8,Finance Leaders,LinkedIn
8,9,Ops Leaders,LinkedIn
9,10,Blog CTA,Organic Search


## 5. Prepare Lead Fact Table

In [9]:
lead_fact = leads.merge(
    campaign_dim,
    left_on=["campaign", "channel"],
    right_on=["campaign_name", "channel"],
    how="left",
    validate="many_to_one",
)

lead_fact = lead_fact[
    [
        "lead_id",
        "created_date",
        "campaign_id",
        "region",
        "company_size",
        "industry",
        "acquisition_cost",
    ]
].copy()

if lead_fact["campaign_id"].isna().any():
    raise ValueError(
        "Some leads could not be mapped to a campaign_id."
    )

lead_fact["campaign_id"] = (
    lead_fact["campaign_id"].astype(int)
)

lead_fact.head()


,lead_id,created_date,campaign_id,region,company_size,industry,acquisition_cost
0,L00052,2025-01-01,14,Canada,Mid-Market,SaaS,10.89
1,L00109,2025-01-01,6,US,SMB,Logistics,66.52
2,L00197,2025-01-01,15,Canada,Mid-Market,Finance,65.79
3,L00327,2025-01-01,6,Canada,SMB,SaaS,96.07
4,L00413,2025-01-02,2,US,Mid-Market,SaaS,13.52


## 6. Prepare Funnel Event Fact Table

In [10]:
event_fact = events[
    [
        "event_id",
        "lead_id",
        "stage",
        "stage_date",
        "revenue",
    ]
].copy()

event_fact.head()


,event_id,lead_id,stage,stage_date,revenue
0,EVT000001,L00001,Lead,2025-03-25,0.0
1,EVT000002,L00001,Lost,2025-04-04,0.0
2,EVT000003,L00002,Lead,2025-06-11,0.0
3,EVT000004,L00002,MQL,2025-06-24,0.0
4,EVT000005,L00002,Lost,2025-07-13,0.0


## 7. Build MySQL Connection URLs

In [11]:
server_url = URL.create(
    drivername="mysql+pymysql",
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=MYSQL_PORT,
)

database_url = URL.create(
    drivername="mysql+pymysql",
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    database=MYSQL_DATABASE,
)

print("Connection configuration created.")


Connection configuration created.


## 8. Test MySQL Server Connection

In [12]:
server_engine = create_engine(
    server_url,
    pool_pre_ping=True,
)

with server_engine.connect() as connection:
    mysql_version = connection.execute(
        text("SELECT VERSION();")
    ).scalar()

print(f"Connected to MySQL {mysql_version}")


Connected to MySQL 8.0.46


## 9. Create Analytics Database

In [13]:
with server_engine.begin() as connection:
    connection.execute(
        text(
            f"CREATE DATABASE IF NOT EXISTS "
            f"{MYSQL_DATABASE} "
            "CHARACTER SET utf8mb4 "
            "COLLATE utf8mb4_unicode_ci;"
        )
    )

print(f"Database '{MYSQL_DATABASE}' is ready.")


Database 'growth_analytics' is ready.


## 10. Connect to the Analytics Database

In [14]:
engine = create_engine(
    database_url,
    pool_pre_ping=True,
)

with engine.connect() as connection:
    active_database = connection.execute(
        text("SELECT DATABASE();")
    ).scalar()

print(f"Connected to: {active_database}")


Connected to: growth_analytics


## 11. Create Database Tables

The schema is stored in `sql/01_schema.sql`.

For repeatable local development, we recreate the Stage 3 tables before loading them.  
This is appropriate for this portfolio project because the source datasets are small and fully reproducible.


In [15]:
schema_sql = SCHEMA_FILE.read_text(
    encoding="utf-8"
)

reset_sql = [
    "SET FOREIGN_KEY_CHECKS = 0",
    "DROP TABLE IF EXISTS fact_funnel_event",
    "DROP TABLE IF EXISTS fact_lead_acquisition",
    "DROP TABLE IF EXISTS dim_campaign",
    "SET FOREIGN_KEY_CHECKS = 1",
]

with engine.begin() as connection:
    for statement in reset_sql:
        connection.execute(text(statement))

    statements = [
        statement.strip()
        for statement in schema_sql.split(";")
        if statement.strip()
    ]

    for statement in statements:
        connection.execute(text(statement))

print("Database schema created successfully.")


Database schema created successfully.


## 12. Load Campaign Dimension

In [16]:
campaign_dim.to_sql(
    name="dim_campaign",
    con=engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=500,
)

print(
    f"Loaded {len(campaign_dim):,} rows "
    "into dim_campaign."
)


Loaded 16 rows into dim_campaign.


## 13. Load Lead Acquisition Fact

In [17]:
lead_fact.to_sql(
    name="fact_lead_acquisition",
    con=engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=500,
)

print(
    f"Loaded {len(lead_fact):,} rows "
    "into fact_lead_acquisition."
)


Loaded 500 rows into fact_lead_acquisition.


## 14. Load Funnel Event Fact

In [18]:
event_fact.to_sql(
    name="fact_funnel_event",
    con=engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000,
)

print(
    f"Loaded {len(event_fact):,} rows "
    "into fact_funnel_event."
)


Loaded 1,445 rows into fact_funnel_event.


## 15. Verify Database Row Counts

In [19]:
verification_query = text("""
    SELECT 'dim_campaign' AS table_name,
           COUNT(*) AS row_count
    FROM dim_campaign

    UNION ALL

    SELECT 'fact_lead_acquisition',
           COUNT(*)
    FROM fact_lead_acquisition

    UNION ALL

    SELECT 'fact_funnel_event',
           COUNT(*)
    FROM fact_funnel_event;
""")

with engine.connect() as connection:
    row_counts = pd.read_sql(
        verification_query,
        connection,
    )

row_counts


,table_name,row_count
0,dim_campaign,16
1,fact_lead_acquisition,500
2,fact_funnel_event,1445


## 16. Verify Expected vs Loaded Rows

In [20]:
expected_counts = pd.DataFrame({
    "table_name": [
        "dim_campaign",
        "fact_lead_acquisition",
        "fact_funnel_event",
    ],
    "expected_rows": [
        len(campaign_dim),
        len(lead_fact),
        len(event_fact),
    ],
})

verification = expected_counts.merge(
    row_counts,
    on="table_name",
    how="left",
)

verification["matches"] = (
    verification["expected_rows"]
    == verification["row_count"]
)

verification


,table_name,expected_rows,row_count,matches
0,dim_campaign,16,16,True
1,fact_lead_acquisition,500,500,True
2,fact_funnel_event,1445,1445,True


In [21]:
if not verification["matches"].all():
    raise ValueError(
        "Database row counts do not match "
        "the processed source data."
    )

print("All row counts match.")


All row counts match.


## 17. Verify Relationships

In [22]:
relationship_query = text("""
    SELECT
        SUM(CASE WHEN c.campaign_id IS NULL THEN 1 ELSE 0 END)
            AS leads_without_campaign,

        (
            SELECT COUNT(*)
            FROM fact_funnel_event e
            LEFT JOIN fact_lead_acquisition l
                ON e.lead_id = l.lead_id
            WHERE l.lead_id IS NULL
        ) AS orphan_events
    FROM fact_lead_acquisition l
    LEFT JOIN dim_campaign c
        ON l.campaign_id = c.campaign_id;
""")

with engine.connect() as connection:
    relationship_check = pd.read_sql(
        relationship_query,
        connection,
    )

relationship_check


,leads_without_campaign,orphan_events
0,0.0,0


## 18. Inspect the Database Model

In [23]:
table_query = text("""
    SELECT
        TABLE_NAME,
        TABLE_ROWS
    FROM information_schema.TABLES
    WHERE TABLE_SCHEMA = :database_name
    ORDER BY TABLE_NAME;
""")

with engine.connect() as connection:
    database_tables = pd.read_sql(
        table_query,
        connection,
        params={"database_name": MYSQL_DATABASE},
    )

database_tables


,TABLE_NAME,TABLE_ROWS
0,dim_campaign,16
1,fact_funnel_event,1445
2,fact_lead_acquisition,500


## 19. Test a Business Join

In [24]:
sample_query = text("""
    SELECT
        l.lead_id,
        l.created_date,
        c.channel,
        c.campaign_name,
        l.region,
        l.company_size,
        l.industry,
        l.acquisition_cost
    FROM fact_lead_acquisition l
    JOIN dim_campaign c
        ON l.campaign_id = c.campaign_id
    ORDER BY l.created_date, l.lead_id
    LIMIT 10;
""")

with engine.connect() as connection:
    sample = pd.read_sql(
        sample_query,
        connection,
    )

sample


,lead_id,created_date,channel,campaign_name,region,company_size,industry,acquisition_cost
0,L00052,2025-01-01,Referral,Partner Referral,Canada,Mid-Market,SaaS,10.89
1,L00109,2025-01-01,Google Ads,Search Campaign,US,SMB,Logistics,66.52
2,L00197,2025-01-01,Webinar,Finance Automation Webinar,Canada,Mid-Market,Finance,65.79
3,L00327,2025-01-01,Google Ads,Search Campaign,Canada,SMB,SaaS,96.07
4,L00413,2025-01-02,Email,Product Update,US,Mid-Market,SaaS,13.52
5,L00292,2025-01-03,Organic Search,Blog CTA,US,SMB,Retail,13.15
6,L00450,2025-01-03,Webinar,Finance Automation Webinar,US,Mid-Market,Manufacturing,51.75
7,L00187,2025-01-04,Referral,Customer Referral,US,Mid-Market,Healthcare,15.95
8,L00256,2025-01-04,Google Ads,Competitor Terms,US,Mid-Market,SaaS,96.97
9,L00309,2025-01-04,Email,Product Update,US,SMB,Finance,27.32


## 20. Final Stage 3 Verification

In [26]:
final_check_query = text("""
    SELECT
        COUNT(DISTINCT l.lead_id) AS total_leads,
        COUNT(DISTINCT CASE
            WHEN e.stage = 'Customer'
            THEN e.lead_id
        END) AS customers,
        ROUND(SUM(DISTINCT 0), 2) AS placeholder
    FROM fact_lead_acquisition l
    LEFT JOIN fact_funnel_event e
        ON l.lead_id = e.lead_id;
""")

# We keep this cell intentionally simple:
# the important analytical metrics will be created in Stage 4.
with engine.connect() as connection:
    final_check = connection.execute(
        text(
            "SELECT "
            "(SELECT COUNT(*) FROM fact_lead_acquisition) AS leads, "
            "(SELECT COUNT(*) FROM fact_funnel_event) AS events, "
            "(SELECT COUNT(*) FROM dim_campaign) AS campaigns;"
        )
    ).mappings().one()

print(f"Leads: {final_check['leads']:,}")
print(f"Events: {final_check['events']:,}")
print(f"Campaigns: {final_check['campaigns']:,}")
print("\ndatabase setup is complete.")


Leads: 500
Events: 1,445
Campaigns: 16

database setup is complete.
